# Kathmandu traffic vision - train plates

Fine-tunes YOLO11s to find **number plates** (one class, all Nepali formats). Data: Kathmandu-valley plate images, Nepali bike-and-car photos, and distant plates from Indian traffic video (weighted x5, closest to a junction camera). Reading the characters is a later step.

In [ ]:
!pip install -q ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:

import os, random, yaml
from pathlib import Path
IN, OUT = Path("/kaggle/input/datasets"), Path("/kaggle/working/data")
VAL_LISTS = {}

def label_of(img):
    parts = list(img.parts)
    if "images" in parts:                                       # YOLO layout: .../images/... -> .../labels/...
        i = len(parts) - 1 - parts[::-1].index("images")
        p = Path(*parts[:i], "labels", *parts[i + 1:]).with_suffix(".txt")
        if p.exists():
            return p
    return img.with_suffix(".txt")                              # some sets keep labels next to images

def add(tag, split, images, names, mapping, repeat=1):
    """Link images into OUT/<split>, rewriting labels: keep only source classes in `mapping`
    (source name -> target id); boxes of other classes are dropped, the image stays."""
    d = OUT / split
    (d / "images").mkdir(parents=True, exist_ok=True)
    (d / "labels").mkdir(parents=True, exist_ok=True)
    n_box, kept = 0, []
    for img in images:
        rows, lab = [], label_of(img)
        if lab.exists():
            for ln in lab.read_text(errors="ignore").splitlines():
                parts = ln.split()
                if len(parts) != 5 or not parts[0].isdigit():      # skip polygons / malformed rows
                    continue
                k = int(parts[0])
                if k < len(names) and names[k] in mapping:
                    rows.append(" ".join([str(mapping[names[k]])] + parts[1:]))
        for r in range(repeat):
            stem = f"{tag}_{r}_{img.parent.name}_{img.stem}".replace(" ", "_")
            link = d / "images" / (stem + img.suffix.lower())
            if not link.exists():
                link.symlink_to(img)
            (d / "labels" / (stem + ".txt")).write_text("\n".join(rows))
            kept.append(str(link))
        n_box += len(rows)
    if split == "val":
        VAL_LISTS[tag] = kept
    print(f"{tag:<10} {split:<5} {len(images):>6} images x{repeat}  {n_box:>7} boxes kept")

def imgs(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

def split_random(images, val_share=0.15, seed=0):
    images = images[:]
    random.Random(seed).shuffle(images)
    n = int(len(images) * val_share)
    return images[n:], images[:n]

def write_yaml(names):
    (OUT / "data.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images", "val": "val/images",
                                                 "names": dict(enumerate(names))}))
    for tag, files in VAL_LISTS.items():          # per-source validation sets, to see where the model is weak
        (OUT / f"val_{tag}.txt").write_text("\n".join(files))
        (OUT / f"data_{tag}.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images",
                                                            "val": f"val_{tag}.txt", "names": dict(enumerate(names))}))


In [ ]:

NAMES = ["plate"]

ish = IN / "ishworsubedii/vehicle-number-plate-datasetnepal/vehicle_number_plate_detection"
tr, va = split_random(imgs(ish / "images"))
add("kathmandu", "train", tr, ["plate"], {"plate": 0})
add("kathmandu", "val", va, ["plate"], {"plate": 0})

nep = IN / "kshitizgajurel042/nepali-bike-and-car-images-with-annotations/Final Data"
nep_names = ["car", "license_plate", "bike"]
add("nepal", "train", imgs(nep / "images/train"), nep_names, {"license_plate": 0}, repeat=2)
add("nepal", "val", imgs(nep / "images/val"), nep_names, {"license_plate": 0})

dee = IN / "deepakat002/indian-vehicle-number-plate-yolo-annotation"
tr, va = split_random(imgs(dee))
add("far_plates", "train", tr, ["number_plate"], {"number_plate": 0}, repeat=5)   # distant plates, like a junction camera
add("far_plates", "val", va, ["number_plate"], {"number_plate": 0})
write_yaml(NAMES)


In [ ]:

import glob, shutil
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
model.train(data=str(OUT / "data.yaml"), epochs=60, time=3.0, imgsz=640, batch=32, patience=12,
            project="/kaggle/working/runs", name="plates", workers=4, seed=0, plots=True)
best = sorted(glob.glob("/kaggle/working/runs/plates*/weights/best.pt"))[-1]
shutil.copy(best, "/kaggle/working/plates_best.pt")
print("saved /kaggle/working/plates_best.pt")


In [ ]:

import pandas as pd
m = YOLO("/kaggle/working/plates_best.pt")
rows = []
for tag in VAL_LISTS:
    r = m.val(data=str(OUT / f"data_{tag}.yaml"), imgsz=640, verbose=False, plots=False)
    row = {"source": tag, "images": len(VAL_LISTS[tag]), "mAP50": round(r.box.map50, 3), "mAP50-95": round(r.box.map, 3)}
    for i, n in enumerate(NAMES):
        if i < len(r.box.ap50):
            row[f"AP50 {n}"] = round(float(r.box.ap50[i]), 3)
    rows.append(row)
res = pd.DataFrame(rows)
res.to_csv("/kaggle/working/plates_per_source.csv", index=False)
print(res.to_string(index=False))
shutil.rmtree(OUT)    # links only; keep the output small
